In [ ]:
#!/usr/bin/env python3
# commander.py
# Run on macOS - sends simultaneous experiment start commands to 4 RPis via SSH
#
# Prerequisites:
#   pip install paramiko
#
# Usage:
#   python commander.py

import paramiko
import threading
import datetime
import time

# ==========================================
# ⚙️ Experiment settings <- user configuration
# ==========================================

# --- Common settings ---
DELAY_SECONDS  = 30
         # Delay between command dispatch and actual start (seconds)
NUM_SAMPLES    = 10000     # number of image samples
INTERVAL_MS    = 50         # image interval (ms)
                            # -> total experiment duration = NUM_SAMPLES x INTERVAL_MS (ms)

CSI_MULTIPLIER = 2          # Collect CSI N times denser than images
                            # Recommend >= 20ms (< 10ms may cause timing errors)
                            # -> CSI interval   = INTERVAL_MS / CSI_MULTIPLIER
                            # -> CSI samples    = NUM_SAMPLES x CSI_MULTIPLIER

# --- Camera settings (image mode fixed) ---
CAM_WIDTH    = 640          # output width (full FOV scaled down to this size)
CAM_HEIGHT   = 480          # output height
CAM_QUALITY  = 80           # JPEG quality (0-100)
CAM_EXPOSURE = 10           # exposure time (ms)
CAM_GAIN     = 2.0          # analogue gain

# ==========================================
# RPi SSH credentials (using hostname.local)
# ==========================================
SSH_USER = "user_name"
SSH_PASS = ""  # TODO: enter password here or manage via environment variable

DEVICES = {
    "cam1"   : "cam1.local",
    "cam2"   : "cam2.local",
    "ap-rpi" : "ap-rpi.local",
    "sta-rpi": "sta-rpi.local",
}

# ==========================================
# Start time calculation
# ==========================================
def get_start_time():
    t = datetime.datetime.now() + datetime.timedelta(seconds=DELAY_SECONDS)
    return t.strftime("%H:%M:%S")

# ==========================================
# Build commands for each RPi
# ==========================================
def build_commands(T):
    csi_interval_ms = INTERVAL_MS // CSI_MULTIPLIER
    csi_num_samples = NUM_SAMPLES * CSI_MULTIPLIER

    cam_cmd = (
        f"cd ~/Raspcam && /home/User/miniforge3/envs/Raspcam/bin/python3 cam_recorder.py"
        f" --start {T}"
        f" --num {NUM_SAMPLES}"
        f" --interval {INTERVAL_MS}"
        f" --width {CAM_WIDTH}"
        f" --height {CAM_HEIGHT}"
        f" --quality {CAM_QUALITY}"
        f" --exposure {CAM_EXPOSURE}"
        f" --gain {CAM_GAIN}"
    )

    return {
        "cam1"   : cam_cmd,
        "cam2"   : cam_cmd,
        "sta-rpi": (
            f"cd ~/esp/VO-CSI/STA/ESP32-CSI_Sta && python3 csi_save.py"
            f" --start {T}"
            f" --num {csi_num_samples}"
            f" --interval {csi_interval_ms}"
        ),
        "ap-rpi" : (
            f"cd ~/esp/AP/ESP32-CSI_AP && python3 ap_trigger.py"
            f" --start {T}"
            f" --num {csi_num_samples}"
            f" --interval {csi_interval_ms}"
        ),
    }

# ==========================================
# SSH execution function (per thread)
# ==========================================
results = {}

def ssh_run(name, host, cmd):
    try:
        client = paramiko.SSHClient()
        client.set_missing_host_key_policy(paramiko.AutoAddPolicy())
        client.connect(host, username=SSH_USER, password=SSH_PASS, timeout=15)

        print(f"[{name}] Connected ({host})")
        print(f"[{name}] Command dispatched 🚀 (log: '{name}_log.txt')")

        transport = client.get_transport()
        channel = transport.open_session()
        channel.get_pty()
        channel.exec_command(cmd)

        with open(f"{name}_log.txt", "w", encoding="utf-8") as log_file:
            while True:
                if channel.recv_ready():
                    data = channel.recv(4096).decode('utf-8', 'ignore')
                    if data:
                        log_file.write(data)
                        log_file.flush()

                if channel.recv_stderr_ready():
                    err_data = channel.recv_stderr(4096).decode('utf-8', 'ignore')
                    if err_data:
                        log_file.write(f"[ERR] {err_data}")
                        log_file.flush()

                if channel.exit_status_ready() and not channel.recv_ready() and not channel.recv_stderr_ready():
                    break

                time.sleep(0.05)

        exit_code = channel.recv_exit_status()
        results[name] = exit_code
        print(f"[{name}] Experiment done, log saved (exit={exit_code})")

        channel.close()
        client.close()

    except Exception as e:
        print(f"[{name}] Error: {e}")
        results[name] = -1

# ==========================================
# Main
# ==========================================
def main():
    T               = get_start_time()
    total_sec       = NUM_SAMPLES * INTERVAL_MS / 1000
    csi_interval_ms = INTERVAL_MS // CSI_MULTIPLIER
    csi_num_samples = NUM_SAMPLES * CSI_MULTIPLIER

    print(f"\n{'='*45}")
    print(f"  Experiment parameters")
    print(f"{'='*45}")
    print(f"  Start time  : {T}  (+{DELAY_SECONDS}s)")
    print(f"  Total time  : {total_sec:.1f}s")
    print(f"  ---")
    print(f"  Camera mode : image (fixed)")
    print(f"  Resolution  : {CAM_WIDTH}x{CAM_HEIGHT}")
    print(f"  Exp/Gain    : {CAM_EXPOSURE}ms / {CAM_GAIN}")
    print(f"  Image intv  : {INTERVAL_MS}ms  ({1000//INTERVAL_MS}Hz)  x {NUM_SAMPLES} samples")
    print(f"  ---")
    print(f"  CSI mult    : x{CSI_MULTIPLIER}")
    print(f"  CSI intv    : {csi_interval_ms}ms  ({1000//csi_interval_ms}Hz)  x {csi_num_samples} samples")
    print(f"{'='*45}\n")

    input("Ready. Press Enter to dispatch commands to all 4 devices...")

    COMMANDS = build_commands(T)

    threads = []
    for name, host in DEVICES.items():
        t = threading.Thread(target=ssh_run, args=(name, host, COMMANDS[name]))
        threads.append(t)

    for t in threads:
        t.start()
    for t in threads:
        t.join()

    print("\n===== All done =====")
    for name, code in results.items():
        status = "✅" if code == 0 else "❌"
        print(f"  {status} {name}: exit={code}")

if __name__ == "__main__":
    main()